> **Disclaimer.** This project is educational and is not investment advice. Backtest and optimizer results are historical and do not predict future results.

# The best trials of the 50,000-trial search, against the paper's rule

The same costs, sizing and development days as `winfut_wdo_optimization.ipynb` (2021-12-01 to 2024-12-30; the final test period is not loaded). The reference rows are the paper's rule and the two variants of that notebook's section 4; the other rows are the best trial of each mode over the **whole development period** (the search's own score, so **in-sample and inflated by the selection**: the leave-one-year-out and luck-baseline sections of that notebook say how far to trust them).

Added here: profit factor, Sortino, Calmar, CAGR, and whether the strategy beats the **CDI** (`docs/cdi.md`: CAGR against the compounded CDI, idle cash earning nothing) and **buying the IBOV** over the same window (CAGR against CAGR, `compare_with_index`). Profit factor is over **daily** returns (sum of winning days over sum of losing days), not per trade.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

from b3core.benchmark import compare_with_cdi, compare_with_index
from b3core.intraday_momentum import extended_performance, performance
from b3core.intraday_search import (
    INSTRUMENTS,
    TrialSet,
    decode,
    encode,
    portfolio_sharpe,
    run_trial,
    scores,
)
from b3core.winfut import FINAL_START, load_bars

pd.options.display.float_format = "{:,.2f}".format
pd.options.display.width = 230
pd.options.display.max_columns = 40
pd.options.display.max_colwidth = 100

N_TRIALS, SEED, SLIPPAGE_TICKS, MIN_TRADES_FULL = 50_000, 20261116, 1.0, 300
saved = np.load(Path("../samples/optimization") / f"trials_{N_TRIALS}_seed{SEED}.npz")
days = pd.DatetimeIndex(saved["days"])
trials = TrialSet(
    indices=saved["indices"], days=days,
    ret={n: saved[f"ret_{n}"] for n in ("WIN", "WDO")},
    sides={n: saved[f"sides_{n}"] for n in ("WIN", "WDO")},
)
bars = {name: load_bars("../samples", symbol=inst.symbol) for name, inst in INSTRUMENTS.items()}
for frame in bars.values():
    assert frame.index.max() < pd.Timestamp(FINAL_START.date()), "a bar from the final test period was loaded"
print(f"{len(trials.indices):,} saved trials, {len(days)} days, {days[0].date()} to {days[-1].date()}")

50,000 saved trials, 770 days, 2021-12-01 to 2024-12-30


## 1. The rows

The paper's rule and its two variants, then the best trial by WIN, by WDO and by the joint score. Each is re-run here with the same code, so the daily returns are the ones of the search.

In [2]:
def clock(minutes: int) -> str:
    return "session end" if minutes >= 1440 else f"{minutes // 60:02d}:{minutes % 60:02d}"


def describe(p: dict) -> str:
    return (f"{p['lookback']}d x{p['band_mult']:g} every {p['trade_freq']}m from {clock(p['first_decision'])} "
            f"exit {clock(p['exit_minute'])} {p['stop']}/{p['stop_check']} {p['sides']}")


PAPER = {"lookback": 14, "band_mult": 1.0, "trade_freq": 30, "first_decision": 540, "exit_minute": 1440,
         "stop": "vwap_band", "stop_check": "grid", "sides": "both"}
EVERYTHING = np.ones(len(days), dtype=bool)
chosen = {"paper's rule": PAPER, "paper, exit at 17:30": PAPER | {"exit_minute": 1050},
          "paper, every-bar stop": PAPER | {"stop_check": "every_bar"}}
for mode in ("WIN", "WDO", "joint"):
    s = scores(trials, EVERYTHING, mode, MIN_TRADES_FULL)
    chosen[f"best by {mode}"] = decode(int(trials.indices[int(np.nanargmax(s))]))
display(pd.Series({k: describe(v) for k, v in chosen.items()}, name="combination").to_frame())

daily: dict[str, dict[str, pd.Series]] = {}
for label, params in chosen.items():
    returns, _ = run_trial(bars, params, days, slippage_ticks=SLIPPAGE_TICKS)
    flat = {n: pd.Series(np.nan_to_num(r, nan=0.0), index=days) for n, r in returns.items()}  # no sigma = flat day
    daily[label] = flat | {"50/50": (flat["WIN"] + flat["WDO"]) / 2.0}
# the search stored the same numbers: check one trial against its stored row
row = int(np.where(trials.indices == encode(chosen["best by joint"]))[0][0])
assert np.allclose(daily["best by joint"]["WIN"].to_numpy(), np.nan_to_num(trials.ret["WIN"][row], nan=0.0))

,combination
paper's rule,14d x1 every 30m from 09:00 exit session end vwap_band/grid both
"paper, exit at 17:30",14d x1 every 30m from 09:00 exit 17:30 vwap_band/grid both
"paper, every-bar stop",14d x1 every 30m from 09:00 exit session end vwap_band/every_bar both
best by WIN,20d x1.5 every 60m from 12:00 exit 18:20 current_band/every_bar both
best by WDO,14d x1 every 15m from 09:30 exit 18:20 vwap_only/grid long
best by joint,20d x1.5 every 15m from 10:30 exit 18:20 vwap_only/grid both


## 2. Benchmarks

The CDI series is the Central Bank's SGS 12 (percent a business day) and the IBOV closes are the project's file. Both cover the window.

In [3]:
ibov = pd.read_csv("../samples/ifr2_universe/ibov.csv", index_col=0, parse_dates=True)["Close"].dropna()
cdi_raw = pd.read_csv("../samples/ifr2_universe/cdi.csv")
cdi = pd.Series(cdi_raw["rate"].to_numpy(), index=pd.to_datetime(cdi_raw["date"], format="%d/%m/%Y"))

probe = extended_performance(daily["paper's rule"]["WIN"])
years = probe["years"]
cdi_cmp = compare_with_cdi(0.0, years, cdi, days)
ibov_cmp = compare_with_index(0.0, years, ibov, days)
print(f"window {years:.2f} calendar years; CDI {cdi_cmp.cdi_cagr:.2%} a year (factor {cdi_cmp.factor:.3f}); "
      f"buying the IBOV {ibov_cmp.cagr:.2%} a year (factor {ibov_cmp.factor:.3f})")

window 3.08 calendar years; CDI 12.02% a year (factor 1.418); buying the IBOV 5.91% a year (factor 1.194)


## 3. The tables

One table per instrument and one for the 50/50 average of the two. `CAGR` compounds the daily returns on the running equity over the calendar years of the window. `vs CDI` and `vs IBOV` are the CAGR minus the benchmark's, in percentage points, with a tick when positive. `Max DD` is the equity drawdown on daily closes (the intraday adverse excursion is not in it).

In [4]:
def metrics_row(returns: pd.Series) -> dict:
    base, extra = performance(returns), extended_performance(returns)
    cdi_vs = compare_with_cdi(extra["cagr"], extra["years"], cdi, days)
    ibov_vs = compare_with_index(extra["cagr"], extra["years"], ibov, days)
    return {
        "total return %": base["total_return"] * 100, "CAGR %": extra["cagr"] * 100, "Sharpe": base["sharpe"],
        "Sortino": extra["sortino"], "Calmar": extra["calmar"], "profit factor": extra["profit_factor"],
        "Max DD %": extra["max_drawdown"] * 100, "hit ratio %": base["hit_ratio"] * 100, "t-stat": base["t_stat"],
        "vs CDI, pts": cdi_vs.excess * 100, "beats CDI": "yes" if cdi_vs.beats_cdi else "no",
        "vs IBOV, pts": ibov_vs.excess * 100, "beats IBOV": "yes" if ibov_vs.beats else "no",
    }


tables = {}
for series in ("WIN", "WDO", "50/50"):
    tables[series] = pd.DataFrame({label: metrics_row(daily[label][series]) for label in chosen}).T
    print(f"=== {series} ===")
    display(tables[series])

=== WIN ===


,total return %,CAGR %,Sharpe,Sortino,Calmar,profit factor,Max DD %,hit ratio %,t-stat,"vs CDI, pts",beats CDI,"vs IBOV, pts",beats IBOV
paper's rule,20.86,6.34,0.46,0.78,0.36,1.10,17.55,39.38,0.81,-5.67,no,0.43,yes
"paper, exit at 17:30",-15.42,-5.29,-0.29,-0.46,-0.24,0.94,21.93,37.28,-0.51,-17.31,no,-11.21,no
"paper, every-bar stop",21.59,6.55,0.52,0.93,0.35,1.11,18.64,34.14,0.91,-5.46,no,0.64,yes
best by WIN,62.75,17.13,2.05,5.20,2.47,2.01,6.93,41.91,3.59,5.11,yes,11.22,yes
best by WDO,-6.77,-2.25,-0.11,-0.18,-0.16,0.97,13.99,38.17,-0.20,-14.27,no,-8.16,no
best by joint,56.47,15.64,1.18,2.13,1.14,1.32,13.71,46.80,2.06,3.63,yes,9.73,yes


=== WDO ===


,total return %,CAGR %,Sharpe,Sortino,Calmar,profit factor,Max DD %,hit ratio %,t-stat,"vs CDI, pts",beats CDI,"vs IBOV, pts",beats IBOV
paper's rule,19.78,6.03,0.44,0.76,0.34,1.09,17.64,40.18,0.78,-5.98,no,0.12,yes
"paper, exit at 17:30",2.40,0.77,0.13,0.21,0.04,1.03,21.69,40.51,0.22,-11.24,no,-5.14,no
"paper, every-bar stop",-25.62,-9.16,-0.59,-0.96,-0.25,0.89,37.14,35.50,-1.03,-21.18,no,-15.07,no
best by WIN,-20.51,-7.18,-0.79,-1.16,-0.24,0.78,29.42,36.82,-1.38,-19.20,no,-13.10,no
best by WDO,87.50,22.64,1.57,3.09,2.55,1.52,8.87,47.71,2.75,10.62,yes,16.73,yes
best by joint,51.12,14.35,1.12,1.93,1.10,1.34,13.08,49.39,1.96,2.33,yes,8.43,yes


=== 50/50 ===


,total return %,CAGR %,Sharpe,Sortino,Calmar,profit factor,Max DD %,hit ratio %,t-stat,"vs CDI, pts",beats CDI,"vs IBOV, pts",beats IBOV
paper's rule,22.22,6.73,0.58,0.97,0.56,1.11,12.01,41.50,1.02,-5.29,no,0.82,yes
"paper, exit at 17:30",-5.64,-1.87,-0.10,-0.16,-0.11,0.98,17.50,39.42,-0.17,-13.88,no,-7.78,no
"paper, every-bar stop",-3.63,-1.19,-0.06,-0.09,-0.06,0.99,19.25,37.61,-0.10,-13.21,no,-7.11,no
best by WIN,14.28,4.43,0.71,1.33,0.35,1.22,12.51,39.47,1.24,-7.59,no,-1.49,no
best by WDO,34.00,9.97,1.06,1.78,1.54,1.24,6.48,45.51,1.86,-2.05,no,4.06,yes
best by joint,55.32,15.37,1.48,2.69,1.87,1.37,8.22,47.11,2.59,3.35,yes,9.45,yes


## 4. Paper's rule against the best trial of the same mode

The change from the reference to its in-sample best, per instrument. A large gain here is what the search is built to find; it is not evidence that the gain holds (see the leave-one-year-out table of the search notebook, where the winners score at or below the paper's rule on the joint score).

In [5]:
pairs = {"WIN": "best by WIN", "WDO": "best by WDO", "50/50": "best by joint"}
rows = {}
for series, best in pairs.items():
    t = tables[series]
    rows[f"{series}: {best}"] = (t.loc[best, ["CAGR %", "Sharpe", "Sortino", "Calmar", "profit factor", "Max DD %", "vs CDI, pts", "vs IBOV, pts"]]
                                 - t.loc["paper's rule", ["CAGR %", "Sharpe", "Sortino", "Calmar", "profit factor", "Max DD %", "vs CDI, pts", "vs IBOV, pts"]])
display(pd.DataFrame(rows).T.rename(columns=lambda c: "change in " + c))

wins = pd.DataFrame({s: [int((tables[s]["beats CDI"] == "yes").sum()), int((tables[s]["beats IBOV"] == "yes").sum())] for s in tables},
                    index=["rows beating the CDI", "rows beating the IBOV"])
print(f"Out of {len(chosen)} rows per table:")
display(wins)

,change in CAGR %,change in Sharpe,change in Sortino,change in Calmar,change in profit factor,change in Max DD %,"change in vs CDI, pts","change in vs IBOV, pts"
WIN: best by WIN,10.79,1.59,4.42,2.11,0.92,-10.62,10.79,10.79
WDO: best by WDO,16.61,1.13,2.34,2.21,0.42,-8.77,16.61,16.61
50/50: best by joint,8.64,0.90,1.72,1.31,0.26,-3.79,8.64,8.64


Out of 6 rows per table:


,WIN,WDO,50/50
rows beating the CDI,2,2,1
rows beating the IBOV,4,3,3
